# Analyses fiables avec BERT — DistilBERT sur `tweet_eval` (sentiment)

Projet de bout en bout : fine-tuning de **DistilBERT** pour la classification de sentiment
(négatif / neutre / positif) sur `tweet_eval`, avec **évaluation + étalonnage**, **introspection
de l'attention**, et une fonction d'inférence explicable `analyze_text()` prête pour la production.

> ⚠️ **À exécuter sur GPU** (Colab : Runtime → GPU). Le fine-tuning sur CPU serait très lent.

## 0. Installation & imports

In [ ]:
%%capture
!pip install -U transformers datasets evaluate scikit-learn matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, AutoModel,
    TrainingArguments, Trainer,
)
from sklearn.metrics import accuracy_score, f1_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device :", device)

## 1. Chargement et inspection des données

In [ ]:
dataset = load_dataset("tweet_eval", "sentiment")
print(dataset)

# Répartition des splits
for split in dataset:
    print(f"{split}: {len(dataset[split])} exemples")

In [ ]:
# Vérifier les 3 étiquettes et leur distribution
label_names = dataset["train"].features["label"].names
print("Étiquettes :", label_names)   # ['negative', 'neutral', 'positive']

import collections
dist = collections.Counter(dataset["train"]["label"])
for label_id, count in sorted(dist.items()):
    print(f"  {label_names[label_id]:8s} : {count} ({100*count/len(dataset['train']):.1f}%)")

In [ ]:
# Enregistrer 2 tweets d'exemple par étiquette (pour l'inspection d'attention plus tard)
examples_by_label = {name: [] for name in label_names}
for row in dataset["train"]:
    name = label_names[row["label"]]
    if len(examples_by_label[name]) < 2:
        examples_by_label[name].append(row["text"])
    if all(len(v) == 2 for v in examples_by_label.values()):
        break

for name, texts in examples_by_label.items():
    print(f"\n=== {name} ===")
    for t in texts:
        print("  -", t)

## 2. Pipeline de tokenisation

`AutoTokenizer` de `distilbert-base-uncased`, troncature/padding à **128 tokens**.

In [ ]:
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def preprocess(batch):
    enc = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )
    enc["labels"] = batch["label"]
    return enc

In [ ]:
# Cartographier tout le dataset, mélanger, formater en torch
encoded = dataset.map(preprocess, batched=True)
encoded = encoded.shuffle(seed=42)
encoded.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print(encoded["train"][0]["input_ids"][:12])
print("Colonnes formatées :", encoded["train"].column_names)

## 3. Réglage fin (fine-tuning)

`AutoModelForSequenceClassification` (3 étiquettes), `Trainer` avec `compute_metrics`.

In [ ]:
id2label = {i: n for i, n in enumerate(label_names)}
label2id = {n: i for i, n in enumerate(label_names)}

model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
).to(device)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro"),
    }

> ℹ️ **Note (transformers 5.x)** : le paramètre s'appelle désormais `eval_strategy`
> (et non plus `evaluation_strategy`). `load_best_model_at_end=True` exige que les stratégies
> d'évaluation et de sauvegarde correspondent (ici « epoch »).

In [ ]:
training_args = TrainingArguments(
    output_dir="distilbert-tweeteval-sentiment",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded["train"],
    eval_dataset=encoded["validation"],
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

# Sauvegarder le meilleur point de contrôle + le tokenizer
trainer.save_model("distilbert-tweeteval-sentiment/best")
tokenizer.save_pretrained("distilbert-tweeteval-sentiment/best")
print("Meilleur modèle sauvegardé.")

## 4. Évaluation et étalonnage

In [ ]:
# Évaluation sur la validation
val_metrics = trainer.evaluate(encoded["validation"])
print("Validation :")
print(f"  accuracy : {val_metrics['eval_accuracy']:.4f}")
print(f"  macro F1 : {val_metrics['eval_f1']:.4f}")

In [ ]:
# Prédictions + scores softmax sur le test
test_output = trainer.predict(encoded["test"])
test_logits = test_output.predictions
test_labels = test_output.label_ids

# softmax -> confiance de la classe prédite
probs = torch.softmax(torch.tensor(test_logits), dim=-1).numpy()
pred_ids = probs.argmax(axis=-1)
confidences = probs.max(axis=-1)

test_acc = accuracy_score(test_labels, pred_ids)
test_f1 = f1_score(test_labels, pred_ids, average="macro")
print(f"Test — accuracy : {test_acc:.4f} | macro F1 : {test_f1:.4f}")

In [ ]:
# Histogramme des scores de confiance (intervalles de 0.1)
plt.figure(figsize=(9, 5))
plt.hist(confidences, bins=np.arange(0, 1.05, 0.1), edgecolor="black", alpha=0.8)
plt.title("Distribution des scores de confiance (softmax max) — test")
plt.xlabel("Confiance de la classe prédite")
plt.ylabel("Nombre de tweets")
plt.xticks(np.arange(0, 1.05, 0.1))
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Confiance moyenne : {confidences.mean():.3f}")
print(f"Confiance médiane : {np.median(confidences):.3f}")

**Commentaire étalonnage :** si la masse de l'histogramme se situe très haut (>0,9) alors
que la précision est nettement plus basse, le modèle est **sur-confiant** (mal calibré). À
l'inverse, des confiances concentrées autour de 0,4–0,6 pour des prédictions souvent correctes
indiquent une **sous-confiance**. Une confiance moyenne proche de la précision réelle traduit un
bon étalonnage. En cas de sur-confiance, on peut appliquer un **temperature scaling** sur les
logits.

## 5. Inspection de l'attention

On récupère les poids d'attention de la **dernière couche**, on moyenne sur les têtes, et on
visualise l'attention dirigée depuis le token `[CLS]` vers chaque token.

> ⚠️ **Important (transformers 5.x)** : l'implémentation par défaut `sdpa` renvoie des attentions
> **vides** avec `output_attentions=True`. Il faut charger le modèle avec
> `attn_implementation="eager"` pour obtenir les vrais poids d'attention.

In [ ]:
# Recharger le modèle fine-tuné en mode "eager" pour exposer les attentions
attn_model = AutoModel.from_pretrained(
    "distilbert-tweeteval-sentiment/best",
    attn_implementation="eager",
).to(device)
attn_model.eval()

# Choisir un tweet d'exemple négatif
sample_text = examples_by_label["negative"][0]
print("Tweet analysé :", sample_text)

enc = tokenizer(sample_text, return_tensors="pt", truncation=True, max_length=128).to(device)
with torch.no_grad():
    out = attn_model(**enc, output_attentions=True)

# Dernière couche : (batch, heads, seq, seq) -> moyenne sur les têtes
last_layer_attn = out.attentions[-1][0]        # (heads, seq, seq)
mean_attn = last_layer_attn.mean(dim=0)        # (seq, seq)
cls_attn = mean_attn[0].cpu().numpy()          # attention DEPUIS [CLS]

tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])

In [ ]:
# Visualisation : barres de l'attention [CLS] -> tokens
plt.figure(figsize=(12, 4))
plt.bar(range(len(tokens)), cls_attn, color="steelblue")
plt.xticks(range(len(tokens)), tokens, rotation=60, ha="right")
plt.title("Attention depuis [CLS] (moyenne sur les têtes, dernière couche)")
plt.ylabel("Poids d'attention")
plt.tight_layout()
plt.show()

# Tokens les plus saillants (hors tokens spéciaux)
special = {tokenizer.cls_token, tokenizer.sep_token, tokenizer.pad_token}
ranked = sorted(
    [(tok, w) for tok, w in zip(tokens, cls_attn) if tok not in special],
    key=lambda x: x[1], reverse=True,
)
print("Tokens les plus attentionnés :")
for tok, w in ranked[:8]:
    print(f"  {tok:15s} {w:.4f}")

**Interprétation (à adapter à votre tweet) :** le token `[CLS]`, dont la représentation sert
à la classification, concentre son attention sur les mots porteurs de sentiment. Par exemple, pour
un tweet négatif, le modèle se focalise typiquement sur des termes comme *terrible*, *worst* ou
*service*, ce qui explique la prédiction « negative ».

## 6. Fonction d'inférence explicable `analyze_text()`

Fonction de type production qui renvoie `{label, confidence, highlighted_tokens}` — prête à être
intégrée à des outils de support.

In [ ]:
# Modèle de classification (pour label+confiance) rechargé en eager pour l'attention
clf_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-tweeteval-sentiment/best",
    attn_implementation="eager",
).to(device)
clf_model.eval()

def analyze_text(text, top_k=5):
    """Renvoie l'étiquette prédite, la confiance, et les tokens les plus contributifs."""
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        out = clf_model(**enc, output_attentions=True)

    # Étiquette + confiance
    probs = torch.softmax(out.logits, dim=-1)[0]
    label_id = int(probs.argmax())
    confidence = float(probs[label_id])

    # Attention [CLS] -> tokens (moyenne sur têtes, dernière couche)
    cls_attn = out.attentions[-1][0].mean(dim=0)[0].cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])

    special = {tokenizer.cls_token, tokenizer.sep_token, tokenizer.pad_token}
    scored = [(tok, float(w)) for tok, w in zip(tokens, cls_attn) if tok not in special]
    highlighted = sorted(scored, key=lambda x: x[1], reverse=True)[:top_k]

    return {
        "label": clf_model.config.id2label[label_id],
        "confidence": round(confidence, 4),
        "highlighted_tokens": highlighted,
    }

In [ ]:
# Démonstration
for txt in [
    "The customer service was absolutely terrible and slow.",
    "I love this new update, it works great!",
    "The package arrived today.",
]:
    result = analyze_text(txt)
    print(f"Texte : {txt}")
    print(f"  -> label={result['label']} | confidence={result['confidence']}")
    print(f"  -> tokens clés : {result['highlighted_tokens']}\n")

## ✅ Livrables

- **Répartition des données** et vérification des 3 classes (partie 1).
- **Pipeline de tokenisation** (128 tokens, `batched=True`, format torch) (partie 2).
- **Point de contrôle DistilBERT fine-tuné** sauvegardé dans
  `distilbert-tweeteval-sentiment/best/` avec tokenizer (partie 3).
- **Rapport d'évaluation** : accuracy + macro F1 + histogramme de confiance avec commentaire
  d'étalonnage (parties 3–4).
- **Introspection de l'attention** : carte de l'attention `[CLS]` avec interprétation (partie 5).
- **`analyze_text()`** renvoyant `{label, confidence, highlighted_tokens}` (partie 6).

Le dossier `distilbert-tweeteval-sentiment/best/` (poids + tokenizer + config) est réutilisable
par vos coéquipiers via `AutoModelForSequenceClassification.from_pretrained(...)`.